# Stage 3 — MATLAB ↔ Python GPU analytical RIS statistics

Bu aşama şu MATLAB zincirini taşır:

\[
\texttt{generate\_eff\_moments}
\rightarrow
\texttt{evaluate\_gamma\_metric}
\]

Kontrol edilenler:

\[
UBR,\quad
\mu_{\rm Feff},\quad
\sigma^2_{\rm Feff},\quad
\mu_{\rm SNR},\quad
C,\quad
\sigma^2_{\rm Wick}.
\]

## GPU tasarımındaki kritik fark

Candidate batch için

\[
G=\gamma\gamma^H
\]

tensorü **oluşturulmuyor**.

Onun yerine

\[
\sum_{ij}\gamma_i\gamma_j^*K_{ij}
=
\gamma^T K\gamma^*
\]

batched matmul ile hesaplanıyor.

Bu sayede `nRIS=512, C=512` için yaklaşık 1 GB'lık
`[C,512,512]` complex tensor oluşturmuyoruz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, json
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

MODULE = ROOT / 'ris_gpu_stats_stage3.py'
if not MODULE.exists():
    MODULE = Path('/content/ris_gpu_stats_stage3.py')

assert MODULE.exists(), f"Modül bulunamadı: {MODULE}"

sys.path.insert(0,str(MODULE.parent))

from ris_gpu_stats_stage3 import (
    compare_stage3_matlab_case,
    build_static_environment,
    prepare_w_state,
    benchmark_gamma_candidates,
)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
print("Loaded:", MODULE)

## MATLAB golden case

MATLAB'da:

```matlab
export_stage3_parity_case( ...
    "stage3_case.mat", ...
    gnb2ris,ris2ue,geometry,c0,lambda_0, ...
    K_BR,K_RU,lsp_BR,lsp_RU);
```

Sonra `stage3_case.mat` dosyasını Colab `/content` altına yükle.

In [ ]:
CASE = Path('/content/stage3_case.mat')
assert CASE.exists(), "stage3_case.mat dosyasını /content altına yükle."
print(CASE)

In [ ]:
# FLOAT64 / COMPLEX128 PARITY
m = compare_stage3_matlab_case(
    str(CASE),
    device='cuda' if torch.cuda.is_available() else 'cpu',
    parity=True,
)

df = pd.DataFrame([m])
display(df.T.rename(columns={0:'value'}))

checks = []
for c in df.columns:
    if c.endswith('_relFro') or c.endswith('_rel'):
        checks.append(float(df[c].iloc[0]))

worst = max(checks)
print("Worst relative error:", worst)

assert worst < 1e-10, (
    f"Stage-3 parity henüz geçmedi: worst={worst:.3e}"
)

print("PASS: Stage-3 MATLAB ↔ Python analytical statistics parity")

In [ ]:
# FLOAT32 / COMPLEX64 PRODUCTION ERROR
m32 = compare_stage3_matlab_case(
    str(CASE),
    device='cuda' if torch.cuda.is_available() else 'cpu',
    parity=False,
)

df32 = pd.DataFrame([m32])
display(df32.T.rename(columns={0:'value'}))

In [ ]:
# OPTIONAL: 512-candidate GPU benchmark using the saved MATLAB environment/W.
from scipy.io import loadmat
import numpy as np

M = loadmat(CASE, squeeze_me=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def sc(name):
    return float(np.asarray(M[name]).reshape(()))

env = build_static_environment(
    rho_RU=M['rhoRU'],
    rho_UR=M['rhoUR'],
    rho_RB=M['rhoRB'],
    rho_BR=M['rhoBR'],
    rho_RUhop=M['rhoRUhop'],
    mu_XPR_BR=sc('muXPR_BR'),
    sigma_XPR_BR=sc('sigmaXPR_BR'),
    mu_XPR_RU=sc('muXPR_RU'),
    sigma_XPR_RU=sc('sigmaXPR_RU'),
    muBR=M['muBR'],
    sigma2BR=sc('sigma2BR'),
    muRU=M['muRU'],
    sigma2RU=sc('sigma2RU'),
    device=dev,
    parity=False,
)

state = prepare_w_state(env,M['W'])

bench = benchmark_gamma_candidates(
    state,
    n_candidates=512,
    repeats=5,
)

print(json.dumps(bench,indent=2))

## Stage 3 geçince

Sonraki adım:

1. RIS response `z -> phi -> beta,gamma`
2. Type-I rank-1 precoder/codebook
3. V3 analytic feature engine

Bunlardan sonra bank + candidate düzeyinde environment'ın ana deterministik
Teacher/Critic veri üretim yolu Python GPU'da tamamlanmış olacak.